# Aşama 1 — Metrik, Yakınsama ve Gradyan Doğrulama

Bu notebook tez bulgusu üretmez. Amaç; ana deneylere geçmeden önce Precision, Recall, F1, AULC, gradyan/aktivasyon ve parametre güncelleme ölçümlerinin teknik olarak çalıştığını doğrulamaktır.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## Güncel public GitHub sürümünü al


In [ ]:
import io, os, shutil, tarfile, requests
from pathlib import Path

OWNER='erdemci07'
REPO='derin-ogrenme-egitim-dinamikleri'
BRANCH='main'
repo_dir=Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root=Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)
r=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{BRANCH}', timeout=60)
r.raise_for_status()
commit_sha=r.json()['sha']
a=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}', timeout=120)
a.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(a.content), mode='r:gz') as tar:
    tar.extractall(extract_root)
kokler=[p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler)!=1:
    raise RuntimeError('GitHub arşiv yapısı beklenenden farklı.')
shutil.move(str(kokler[0]), str(repo_dir))
os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT']=commit_sha
print('Repo hazır:', commit_sha)


## Google Drive yolları


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK=Path('/content/drive/MyDrive')
DRIVE_VERI=DRIVE_KOK/'tez_datasets'
DRIVE_PROJE=DRIVE_KOK/'tez_deney_sonuclari'
DRIVE_AYAR=DRIVE_PROJE/'ortak_ayarlar'
DRIVE_CIKTI=DRIVE_PROJE/'asama1_metrik_yakinsama_gradyan'
if not DRIVE_VERI.is_dir():
    raise FileNotFoundError(f'Veri klasörü bulunamadı: {DRIVE_VERI}')
DRIVE_AYAR.mkdir(parents=True, exist_ok=True)
DRIVE_CIKTI.mkdir(parents=True, exist_ok=True)
SPLIT_DOSYASI=DRIVE_AYAR/'cifar10_split.json'
STATS_DOSYASI=DRIVE_AYAR/'cifar10_stats.json'
os.environ['TEZ_VERI_KLASORU']=str(DRIVE_VERI)
os.environ['TEZ_SPLIT_DOSYASI']=str(SPLIT_DOSYASI)
os.environ['TEZ_STATS_DOSYASI']=str(STATS_DOSYASI)
print('Veri:', DRIVE_VERI)
print('Çıktı:', DRIVE_CIKTI)


## Bağımlılık ve CUDA kontrolü


In [ ]:
subprocess.run(['python','-m','pip','install','-q','-r','requirements_colab.txt'], check=True)
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('T4/CUDA aktif değil.')


## 3 epoch teknik doğrulama

ReLU + SGD ile üç epoch çalıştırılır. 1. ve 3. epochlarda sabit training probe batch üzerinde gradyan ve aktivasyon ölçümü; finalde kopya model üzerinde tek-adımlık parametre güncelleme ölçümü yapılır.


In [ ]:
import importlib
import dogrula_metrik_yakinsama_gradyan as m
importlib.reload(m)
m.ana([
    '--veri-klasoru',str(DRIVE_VERI),
    '--split-dosyasi',str(SPLIT_DOSYASI),
    '--stats-dosyasi',str(STATS_DOSYASI),
    '--cikti-klasoru',str(DRIVE_CIKTI),
    '--epoch','3',
    '--batch-size','128',
    '--probe-boyutu','128',
    '--num-workers','0',
    '--split-seed','2026',
    '--seed','42',
])


## Sonucu özetle


In [ ]:
import json
sonuc_yolu=DRIVE_CIKTI/'metrik_yakinsama_gradyan_dogrulama.json'
s=json.loads(sonuc_yolu.read_text(encoding='utf-8'))
print('GEÇTİ:',s['gecti'])
print('Yakınsama:',s['yakinsama'])
print('Final macro F1:',s['epoch_kayitlari'][-1]['macro_f1'])
print('Final global gradient L2:',s['gradyan_probe'][-1]['global_gradient_l2'])
print('Global relative update:',s['parametre_guncelleme_probe']['global_relative_update'])
print('Kontroller:',s['kontroller'])
print('Dosya:',sonuc_yolu)
